# Nghiên cứu SMS sinh bằng LLM: làm lại phần thiết kế và kiểm từng con số

Nghiên cứu này hỏi: một bộ lọc tin nhắn lừa đảo tiếng Việt chỉ học từ tin do mô hình ngôn ngữ lớn (LLM) viết thì có nhận ra tin lừa đảo thật
không. Mọi quyết định thiết kế đã được đăng ký trước (pre-registration) khi chưa sinh tin nào. Notebook này làm lại phần thiết kế đó từ dữ liệu
công khai, để ai cũng kiểm được rằng các con số trong bản đăng ký không được chọn sau khi nhìn kết quả.

Notebook chạy độc lập. Mọi hàm đều viết ngay trong notebook, không gọi file nào khác của repo. Lần chạy đầu cần mạng để tải corpus.

### Cách chạy

1. Cài thư viện: `pandas`, `numpy`, `scipy`, `scikit-learn`, `huggingface_hub` (Python 3.12).
2. Mở notebook và chạy hết (Run All). Lần đầu mất khoảng một phút.

Notebook tự tải corpus `trannguyenthaituan/vietnamese_sms_dataset` (Hugging Face, CC-BY-4.0), ghim đúng revision `a90e2bd7`. Mọi file tải về
và file trung gian nằm trong thư mục `synthetic_work/` cạnh notebook.

### Notebook kiểm gì

| Bước | Nội dung | So với |
|---|---|---|
| 1 | Corpus thô, khử trùng lặp, mức độ nhãn đi theo lô | các con số ghi trong bản đăng ký |
| 2 | Gom tin gần trùng thành cụm, gán mỗi cụm vào lô đa số | các con số ghi trong bản đăng ký |
| 3 | Rút design fold (khoảng 20% corpus, chỉ dùng để thiết kế kịch bản) | `design_fold_coding.csv` |
| 4 | Mã hoá kịch bản và tính tỷ lệ kịch bản, có sàn 3% | `design_fold_coding.csv`, `scenario_shares.csv` |
| 5 | Dựng danh sách 18.000 yêu cầu sinh tin (gồm hai biến thể prompt) | `requests.csv` |
| 5b | Ghép prompt hoàn chỉnh cho vài yêu cầu, đúng như khi gửi cho mô hình | `prompts_v1.md` |
| 6, 7 | Sinh tin, chia fold đánh giá, huấn luyện và kiểm giả thuyết | chưa chạy: chưa có dữ liệu sinh |

Mỗi bước có phần giải thích, phần code làm lại, và phần kiểm. Cuối mỗi bước có dòng `[bước] k/n checks match`. Bảng cuối notebook gom mọi phép
kiểm.

Notebook không in nội dung tin nhắn nào ra. File được lưu không kèm output, trừ Bước 5b: output của bước này chỉ có prompt và mã băm,
nên được giữ lại để đọc prompt mà không cần chạy.

## Cấu hình: biến môi trường

Bước 1 đến 5 **không cần key nào**: corpus trên Hugging Face là công khai. Các biến dưới đây đều tuỳ chọn, trừ `OPENROUTER_API_KEY` sẽ
cần cho Bước 6 khi phần sinh tin được thêm vào.

| Biến | Dùng để | Mặc định |
|---|---|---|
| `PHISHVN_WORK_DIR` | thư mục chứa corpus tải về và file trung gian | `synthetic_work/` cạnh notebook |
| `PHISHVN_EXPECTED_DIR` | thư mục chứa các file kỳ vọng (`design_fold_coding.csv`, `scenario_shares.csv`, `requests.csv`, `prompts_v1.md`) | thư mục cha của notebook, hoặc `expected/` cạnh notebook |
| `HF_TOKEN` | token Hugging Face, chỉ cần khi tải bị giới hạn lượt | không dùng |
| `OPENROUTER_API_KEY` | gọi các mô hình sinh tin (Bước 6, chưa có) | không dùng ở Bước 1 đến 5 |

Có hai cách đặt biến:
1. Đặt trong shell trước khi mở Jupyter, ví dụ `export PHISHVN_WORK_DIR=/duong/dan`.
2. Tạo file `.env` cạnh notebook, mỗi dòng một biến dạng `TEN=gia_tri`. File này đã được gitignore. Biến đặt trong shell được ưu tiên hơn
   biến trong `.env`.

**Đừng gõ key trực tiếp vào ô code.** Notebook có thể được chia sẻ hoặc commit, và key sẽ đi theo. Ô dưới chỉ in ra biến nào đã được đặt,
không in giá trị của key.

In [ ]:
import os
from pathlib import Path

def load_env_file(path):
    # KEY=value lines; a variable already set in the shell wins
    if not path.exists():
        return []
    loaded = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k not in os.environ:
                os.environ[k] = v
                loaded.append(k)
    return loaded

print("loaded from .env:", load_env_file(Path.cwd() / ".env") or "none")

CONFIG = {
    "PHISHVN_WORK_DIR": os.environ.get("PHISHVN_WORK_DIR", ""),
    "PHISHVN_EXPECTED_DIR": os.environ.get("PHISHVN_EXPECTED_DIR", ""),
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "OPENROUTER_API_KEY": os.environ.get("OPENROUTER_API_KEY", ""),
}
SECRET = {"HF_TOKEN", "OPENROUTER_API_KEY"}
for k, v in CONFIG.items():
    shown = ("set" if v else "not set") if k in SECRET else (v or "(default)")
    print(f"{k:22s} {shown}")

## Bước 0: Thiết lập

Ô đầu khai báo đường dẫn, tải corpus và tìm thư mục chứa các file kỳ vọng. Các file kỳ vọng là những file đã được commit cùng bản đăng ký:
`design_fold_coding.csv`, `scenario_shares.csv`, `requests.csv` và `prompts_v1.md`. Notebook chỉ đọc chúng để so, trừ `prompts_v1.md`
là đầu vào của Bước 5 (danh sách giá trị của các trường văn phong nằm trong file này). Nếu notebook được chạy độc lập ở ngoài repository,
nó sẽ tự động tải các file kỳ vọng này từ kho mã nguồn công khai `vuthainguyen1602/phishvn` về máy.

Ô thứ hai định nghĩa hai hàm kiểm:
- `check(step, name, mine, saved)` ghi lại một phép so. Hai giá trị phải bằng nhau tuyệt đối.
- `report(step)` in số phép kiểm khớp của một bước, kèm chi tiết từng chỗ lệch nếu có.

In [ ]:
import os, re, csv, io, json, random, hashlib, unicodedata, warnings, urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import CountVectorizer
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

HF_DATASET = "trannguyenthaituan/vietnamese_sms_dataset"
HF_REV = "a90e2bd7e8df2376939658075ba8094dcee488ad"
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/sms_synthetic/expected"
EXPECTED_FILES = ["design_fold_coding.csv", "scenario_shares.csv", "requests.csv", "prompts_v1.md"]

def fetch(url, dest, retries=3):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PhishVN walkthrough notebook)"})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                dest.write_bytes(r.read())
                return
        except Exception:
            if attempt == retries - 1:
                raise
            import time
            time.sleep(1 + attempt)

WORK = Path(CONFIG["PHISHVN_WORK_DIR"] or Path.cwd() / "synthetic_work")   # downloads and every file this notebook writes
WORK.mkdir(parents=True, exist_ok=True)
SRC = Path(hf_hub_download(HF_DATASET, "full_dataset.csv", repo_type="dataset", revision=HF_REV,
                           local_dir=WORK / "corpus", token=CONFIG["HF_TOKEN"] or None))

# The expected files sit next to the notebook's folder in the working repository, or in expected/
# Where the expected files are: the variable if set, then expected/ next to the notebook, then the
# study folder of the repository, found from wherever Jupyter was started (the notebook's folder,
# the repository root, or anything in between)
candidates = [Path(CONFIG["PHISHVN_EXPECTED_DIR"])] if CONFIG["PHISHVN_EXPECTED_DIR"] else []
candidates += [Path.cwd() / "expected", Path.cwd().parent]
for up in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(up / "papers" / "future_sms_synthetic")
EXPECTED = next((p for p in candidates if (p / "design_fold_coding.csv").exists()), None)
if EXPECTED is None or not all((EXPECTED / f).exists() for f in EXPECTED_FILES):
    EXPECTED = WORK / "expected"
    EXPECTED.mkdir(exist_ok=True)
    for f in EXPECTED_FILES:
        if not (EXPECTED / f).exists():
            print(f"Downloading {f} from public repository...")
            fetch(f"{EXPECTED_URL}/{f}", EXPECTED / f)

print("corpus:", SRC)
print("expected files read from:", EXPECTED)

In [ ]:
CHECKS = []

def check(step, name, mine, saved):
    # Record one exact comparison between a value computed here and the registered value
    ok = bool(mine == saved)
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": saved, "match": ok})
    return ok

def report(step):
    # Per-step tally, with every mismatch in full
    rows = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows if not c["match"]]
    print(f"[{step}] {len(rows) - len(bad)}/{len(rows)} checks match")
    for c in bad:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

## Bước 1: Corpus thô và mức độ nhãn đi theo lô

File `full_dataset.csv` có các cột `message_id`, `message`, `label` (0 là tin hợp lệ, gọi là ham; 1 là tin dương tính) và `date`. Nhãn dương
tính của nhà phát hành gồm cả spam lẫn lừa đảo, rộng hơn phishing. Nghiên cứu giữ nguyên nhãn này.

Bước này làm ba việc:
- Đếm tin và nhãn, rồi bỏ tin trùng nội dung y hệt (giữ bản đầu tiên). Mọi bước sau làm trên corpus đã khử trùng lặp.
- Đổi cột `date` (dạng ngày/tháng/năm) sang dạng `YYYY-MM-DD`. Cột này là ngày nhập lô dữ liệu, không phải ngày nhận tin, nên ở đây gọi nó
  là **lô** (batch). Tin không có ngày gom vào một nhóm `undated`.
- Đo mức độ nhãn đi theo lô. Bản đăng ký ghi: trong 21 lô có ngày, 14 lô có ít nhất 95% tin cùng một lớp, và hai lô 22/7 và 24/7/2026 chứa
  528 trong 798 tin dương tính. Đây là lý do mọi fold sau này phải chia theo lô: chia ngẫu nhiên thì mô hình học được "tin này thuộc lô nào"
  thay vì "tin này có phải lừa đảo không".

In [ ]:
raw = pd.read_csv(SRC, encoding="utf-8-sig")

def iso(d):
    # 'd/m/Y' -> 'YYYY-MM-DD'; a missing date becomes its own group
    if pd.isna(d):
        return "undated"
    day, month, year = str(d).split("/")
    return f"{int(year):04d}-{int(month):02d}-{int(day):02d}"

raw["batch"] = raw["date"].map(iso)
print("raw messages:", len(raw), "| ham:", int((raw.label == 0).sum()), "| positive:", int((raw.label == 1).sum()))

d = raw.drop_duplicates("message").reset_index(drop=True)
print("after exact-text deduplication:", len(d), "| ham:", int((d.label == 0).sum()),
      "| positive:", int((d.label == 1).sum()))

# Batch x class on the raw corpus, as the registration describes it
tab = pd.crosstab(raw["batch"], raw["label"]).rename(columns={0: "ham", 1: "positive"})
dated = tab.drop(index="undated", errors="ignore")
majority_share = dated.max(axis=1) / dated.sum(axis=1)
two_july = int(tab.loc[["2026-07-22", "2026-07-24"], "positive"].sum())
print("dated batches:", len(dated), "| batches at least 95% one class:", int((majority_share >= 0.95).sum()))
print("positives in the batches of 22 and 24 July 2026:", two_july, "of", int(tab["positive"].sum()))
tab.assign(majority_share=majority_share.round(3))

In [ ]:
S = "1 corpus"
check(S, "raw messages", len(raw), 2991)
check(S, "raw ham", int((raw.label == 0).sum()), 2193)
check(S, "raw positive", int((raw.label == 1).sum()), 798)
check(S, "deduplicated messages", len(d), 2630)
check(S, "deduplicated ham", int((d.label == 0).sum()), 1919)
check(S, "deduplicated positive", int((d.label == 1).sum()), 711)
check(S, "dated batches", len(dated), 21)
check(S, "batches at least 95% one class", int((majority_share >= 0.95).sum()), 14)
check(S, "positives in 22 and 24 July", two_july, 528)
report(S)

## Bước 2: Gom tin gần trùng và gán cụm vào lô

Tin trùng y hệt đã bị bỏ ở Bước 1, nhưng corpus còn nhiều tin gần giống nhau, ví dụ cùng một mẫu quảng cáo chỉ khác một con số. Nếu một tin
nằm ở tập huấn luyện và bản gần giống của nó nằm ở tập test, điểm test sẽ cao hơn thực tế.

Cách gom:
- Mỗi tin được biểu diễn bằng tập các chuỗi 5 ký tự liên tiếp (character 5-gram, sau khi chuyển chữ thường).
- Hai tin được nối với nhau nếu độ giống Jaccard của hai tập này từ 0,8 trở lên. Jaccard bằng số 5-gram chung chia cho số 5-gram của cả hai.
- Mỗi thành phần liên thông của đồ thị này là một **cụm**.
- Một cụm có thể trải trên nhiều lô. Khi đó cả cụm được gán vào lô chiếm đa số trong cụm. Kết quả gọi là **nhóm** (group). Mọi fold sau
  này là hợp của các nhóm nguyên vẹn, nên không cặp tin gần trùng nào bị tách ra hai phía.

Bản đăng ký ghi: 2.079 cụm, và 231 tin bị chuyển sang lô đa số của cụm.

In [ ]:
def components(texts, threshold=0.8):
    # Connected components of the graph joining messages at character 5-gram Jaccard >= threshold
    X = CountVectorizer(analyzer="char", ngram_range=(5, 5), binary=True).fit_transform(texts)
    X = X.astype(np.float32)
    inter = (X @ X.T).toarray()                      # shared 5-grams for every pair
    size = np.asarray(X.sum(axis=1)).ravel()
    jac = inter / np.maximum(size[:, None] + size[None, :] - inter, 1)
    return connected_components(jac >= threshold, directed=False)[1]

d["component"] = components(d["message"].astype(str).tolist())
major = {c: Counter(g["batch"]).most_common(1)[0][0] for c, g in d.groupby("component")}
d["group"] = d["component"].map(major)
moved = int((d["group"] != d["batch"]).sum())
print("components:", d["component"].nunique(), "| messages moved to their component's majority batch:", moved)
print("largest components (messages):", d["component"].value_counts().head(5).tolist())

In [ ]:
S = "2 grouping"
check(S, "components", int(d["component"].nunique()), 2079)
check(S, "messages moved to the majority batch", moved, 231)
report(S)

## Bước 3: Rút design fold

Để viết prompt sinh tin, cần biết tin lừa đảo và tin hợp lệ thật thường theo những kịch bản nào. Nếu đọc cả corpus để biết điều đó thì chính
dữ liệu dùng để chấm điểm đã định hình dữ liệu huấn luyện. Vì vậy, trước mọi việc khác, khoảng 20% corpus được tách ra làm **design fold**:
chỉ fold này được đọc để xây kịch bản, và nó không bao giờ được chấm điểm, huấn luyện hay dùng để chọn ngưỡng.

Luật rút, ghi trong bản đăng ký:
- Hai lô 22/7 và 24/7/2026 giữ lại cho đánh giá, vì chúng chứa phần lớn tin dương tính.
- Các nhóm còn lại được xếp theo thứ tự `random.Random(20261101)` xáo trộn (danh sách nhóm sắp xếp theo tên trước khi xáo).
- Duyệt lần lượt: nhóm nào làm fold vượt 25% số tin thì bỏ qua, và dừng khi fold đạt ít nhất 20%.

Bản đăng ký ghi kết quả: 7 nhóm, 558 tin (21,2%), 71 tin dương tính và 487 tin ham. Phần kiểm còn so danh sách `message_id` của fold, đúng
thứ tự, với file mã hoá đã commit.

In [ ]:
SEED, LO, HI = 20261101, 0.20, 0.25
KEEP_FOR_EVAL = {"2026-07-22", "2026-07-24"}

n = len(d)
sizes = d["group"].value_counts().to_dict()
order = sorted(g for g in sizes if g not in KEEP_FOR_EVAL)
random.Random(SEED).shuffle(order)
chosen, total = [], 0
for g in order:
    if total >= LO * n:
        break
    if total + sizes[g] > HI * n:
        continue                                     # would overshoot 25%: skip this group
    chosen.append(g)
    total += sizes[g]

d["design"] = d["group"].isin(chosen)
fold = d[d["design"]].sort_values(["label", "group", "message_id"]).reset_index(drop=True)
print("design-fold groups, in draw order:", chosen)
print(f"design fold: {len(fold)} messages ({len(fold) / n:.1%}), {int(fold.label.sum())} positive, "
      f"{int((fold.label == 0).sum())} ham")

In [ ]:
coding = pd.read_csv(EXPECTED / "design_fold_coding.csv")

S = "3 design fold"
check(S, "groups drawn", len(chosen), 7)
check(S, "messages", len(fold), 558)
check(S, "share of the corpus, %", round(100 * len(fold) / n, 1), 21.2)
check(S, "positive", int(fold.label.sum()), 71)
check(S, "ham", int((fold.label == 0).sum()), 487)
check(S, "two July batches stay out", bool(not (set(chosen) & KEEP_FOR_EVAL)), True)
check(S, "message ids, in order, equal the committed coding file", fold["message_id"].tolist(), coding["message_id"].tolist())
check(S, "labels equal the committed coding file", fold["label"].tolist(), coding["label"].tolist())
report(S)

## Bước 4: Mã hoá kịch bản và tính tỷ lệ kịch bản

Mỗi tin trong design fold được gán đúng một kịch bản. Danh sách kịch bản ban đầu được viết trước khi đọc fold, và chỉ thêm kịch bản mới khi
có ít nhất ba tin cần đến nó. Có hai kịch bản được thêm theo luật này: đòi nợ đe doạ (`debt_collection_threat`) và tin cờ bạc bị chèn ký tự
nhiễu (`obfuscated_filler`).

**4a. Tin dương tính được mã hoá bằng tay, từng tin một.** Đây là quyết định của người mã hoá, không phải phép tính, nên notebook đọc nó từ
file đã commit và chỉ kiểm tính đầy đủ: tin nào cũng có kịch bản, và chỉ có hai tin nằm ngoài danh sách (gán `other`, không được sinh).

**4b. Tin ham được mã hoá bằng một dãy luật biểu thức chính quy**, áp lên văn bản đã chuyển chữ thường và bỏ dấu, luật nào khớp trước thì
thắng. Có ba ngoại lệ được quyết bằng tay, ghi theo `message_id`: chín tin cá nhân, hai tin quảng cáo nhà mạng mà luật không bắt được, và một
tin bị nhà phát hành gán nhãn ham nhưng đọc là đe doạ đòi nợ (bị loại khỏi tỷ lệ ham). Notebook chạy lại toàn bộ luật này và so với file đã
commit, từng tin.

In [ ]:
def fold_text(s):
    # lower case, diacritics removed (NFD, combining marks dropped), as the ham rules expect
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn").replace("đ", "d")

HAM_RULES = [  # first match wins
    ("otp_code", r"\botp\b|ma xac (minh|thuc|nhan)|xac minh cua ban|verification code|la ma xac|authenticator|ma kich hoat|ma whatsapp|ma xac nhan|nhap ma \[number\]"),
    ("transaction_alert", r"da duoc thanh toan|vua chi tieu|the vpbank .* tang|da duoc nap|thu goc|xu ly xong yeu cau chuyen khoan|duoc thanh toan \[money\] cuoc"),
    ("bill_reminder", r"tong cuoc dv internet|hoa don vi tra sau|cuoc dich vu internet chua"),
    ("subscriber_verification", r"xac thuc (thong tin )?thue bao|thong tu 08|chua xac thuc|cmnd dang ky thue bao|thay doi giay to|cap nhat thong tin thue bao|xac thuc chinh chu|xac thuc/thay doi|cap nhat thanh cong thong tin thue bao|khi xac thuc thong tin|xac thuc de tranh"),
    ("registered_advertising_qc", r"^\s*[\[(]qc[\])]"),
    ("public_service_notice", r"de nghi (nguoi dan|cong dan|toan dan)|bau cu|bo van hoa|an ninh mang va pctp|canh bao\]|thanh pho khuyen nghi|ho so .* (da duoc tiep nhan|da hoan thanh|da xu ly xong)|thoi tiet|hanh dong phong, chong ma tuy|ngay toan dan tiet kiem|dinh danh dien tu cua cong dan|khao sat|mang mu bao hiem"),
    ("appointment_health", r"lich kham|tai kham|ket qua xn"),
    ("delivery_status", r"don hang \[number\] da den|nhan hang trong ngay"),
    ("account_security_notice", r"tam khoa do dang nhap sai|smart otp|kiem tra diem tin dung cic|tuyet doi khong cung cap ma"),
    ("carrier_promo", r"^\s*(\(tb\)|\[tb\]|\(cskh\))"),          # promotions with the carrier's prefix, before account status
    ("carrier_account_status", r"gia han|dang ky thanh cong|dang ky goi .* khong thanh cong|khong du|sap het|ung truoc|ung \[|ung goi|duoc cong them|cuoc vuot goi|muc cuoc khong co goi|cu phap khong hop le|da bi huy|da huy goi|goi cho ban nhung|het luu luong|dong cuoc truoc|het thoi gian su dung|dung luong cua quy khach da het|thoi han goi di|tai khoan cong chua|loi nhan thoai|chan dich vu co dinh|dieu chinh phuong thuc tinh cuoc|tu goi tong dai|ca khi het tien|ma du thuong"),
    ("carrier_promo", r"\(tb\)|\[tb\]|\(cskh\)|soan .* gui (191|888|9)|nap the|vinaphone|viettel|vnpt|data|ctkm"),
]
PERSONAL = {"COM_357", "COM_942", "COM_974", "COM_976", "COM_983", "COM_991", "COM_2227", "COM_2272", "COM_2274"}
CARRIER_PROMO_BY_HAND = {"COM_1286", "COM_2256"}
MISLABELLED = {"COM_1437": "debt_collection_threat"}     # labelled ham by the publisher

def code_ham(message_id, text):
    if message_id in MISLABELLED:
        return MISLABELLED[message_id]
    if message_id in PERSONAL:
        return "personal"
    if message_id in CARRIER_PROMO_BY_HAND:
        return "carrier_promo"
    t = fold_text(text)
    for name, rx in HAM_RULES:
        if re.search(rx, t):
            return name
    return ""

ham = fold[fold.label == 0]
mine_ham = [code_ham(m, t) for m, t in zip(ham["message_id"], ham["message"])]
saved_ham = coding.loc[coding.label == 0, "scenario"].tolist()
diff = [(m, a, b) for m, a, b in zip(ham["message_id"], mine_ham, saved_ham) if a != b]
print("ham messages coded by the rules:", len(mine_ham), "| differing from the committed coding:", len(diff))
print(pd.Series(mine_ham).value_counts().to_string())

In [ ]:
pos = coding[coding.label == 1]
S = "4a positive coding"
check(S, "every positive has a scenario", bool(pos["scenario"].fillna("").ne("").all()), True)
check(S, "positives coded 'other' (not generated)", int(pos["scenario"].str.startswith("other").sum() +
      pos["scenario"].eq("authority_impersonation").sum()), 2)
check(S, "gambling promotions", int((pos.scenario == "gambling_promo").sum()), 37)
check(S, "debt-collection threats (added under the three-message rule)", int((pos.scenario == "debt_collection_threat").sum()), 9)
check(S, "obfuscated filler (added under the three-message rule)", int((pos.scenario == "obfuscated_filler").sum()), 6)
report(S)

S = "4b ham coding"
check(S, "ham coded by the rules equals the committed coding, message by message", len(diff), 0)
check(S, "every ham message has a scenario", all(mine_ham), True)
report(S)

### 4c. Tỷ lệ kịch bản, có sàn 3%

Tỷ lệ của mỗi kịch bản trong một lớp tính như sau:
- Chỉ tính các kịch bản sẽ được sinh: bỏ hai tin `other` ở lớp dương tính và tin ham bị gán nhãn sai.
- Mọi kịch bản có **sàn 3%**, kể cả kịch bản không có tin nào trong fold, để không kịch bản nào vắng mặt khỏi bộ tin sinh. Kịch bản nào có
  tỷ lệ thô dưới 3% thì được nâng lên đúng 3%.
- Lưu ý: bản đăng ký ghi sàn cho "mọi kịch bản trong danh sách ban đầu", nhưng file tỷ lệ đã commit áp sàn cho mọi kịch bản dưới 3%, kể cả
  `account_security_notice`, một kịch bản ham được thêm vào dù chỉ có 2 tin (luật thêm kịch bản đòi ít nhất 3). Notebook làm lại đúng cách
  file đã commit được tính, và ghi điểm lệch này ra ở ô kiểm bên dưới.
- Phần còn lại (100% trừ tổng các sàn) chia cho các kịch bản còn lại theo số tin của chúng trong fold.

Kết quả làm tròn một chữ số thập phân và so với cột `share_pct` của `scenario_shares.csv`. Cột `per_generator` của file đó là tỷ lệ chưa làm
tròn nhân 750, làm tròn từng dòng. Vì làm tròn từng dòng, cột này cộng lại không đúng 750 (lớp dương tính 744, lớp ham 748), nên danh sách yêu
cầu ở Bước 5 không dùng nó mà phân bổ lại từ `share_pct` theo phương pháp phần dư lớn nhất. Notebook vẫn tính lại cột này để kiểm.

In [ ]:
SEED_POS = ["bank_account_verification", "parcel_delivery", "traffic_fine", "identity_vneid", "sim_subscriber",
            "tax_einvoice", "social_insurance", "prize_lucky_draw", "loan_offer", "job_task", "investment",
            "gambling_promo", "reward_points_expiry", "impersonated_relative"]
SEED_HAM = ["otp_code", "transaction_alert", "carrier_promo", "delivery_status", "appointment_health",
            "public_service_notice", "order_confirmation", "registered_advertising_qc", "bill_reminder", "personal"]
ADDED = {"positive": ["debt_collection_threat", "obfuscated_filler"],
         "ham": ["carrier_account_status", "subscriber_verification", "account_security_notice"]}
FLOOR = 3.0

def shares_with_floor(counts, seed_list, added):
    # counts: scenario -> design-fold messages. Seed scenarios below the floor are raised to it,
    # the rest of the mass is split over the other scenarios in proportion to their counts.
    names = list(dict.fromkeys(seed_list + added))
    total = sum(counts.get(s, 0) for s in names)
    raw_share = {s: 100 * counts.get(s, 0) / total for s in names}
    floored = {s for s in names if raw_share[s] < FLOOR}      # seed or added, as the committed shares were computed
    rest = 100 - FLOOR * len(floored)
    free = sum(counts.get(s, 0) for s in names if s not in floored)
    return {s: FLOOR if s in floored else rest * counts.get(s, 0) / free for s in names}, counts

rows = []
for cls, lab, seed in (("positive", 1, SEED_POS), ("ham", 0, SEED_HAM)):
    cnt = Counter(coding.loc[coding.label == lab, "scenario"])
    if lab == 0:
        cnt.pop("debt_collection_threat", None)           # the mislabelled ham message is left out
    sh, _ = shares_with_floor(cnt, seed, ADDED[cls])
    for s, v in sh.items():
        rows.append({"class": cls, "scenario": s, "n_design": cnt.get(s, 0), "share_pct": round(v, 1),
                     "per_generator": round(v * 7.5)})
mine_sh = pd.DataFrame(rows)
saved_sh = pd.read_csv(EXPECTED / "scenario_shares.csv")
both = saved_sh.merge(mine_sh, on=["class", "scenario"], suffixes=("_saved", "_notebook"), how="outer")
both

In [ ]:
S = "4c scenario shares"
check(S, "same scenarios in both tables", len(both) == len(saved_sh) == len(mine_sh), True)
for col in ("n_design", "share_pct", "per_generator"):
    bad = both[both[f"{col}_saved"] != both[f"{col}_notebook"]]
    check(S, f"{col} equal on every row", len(bad), 0)
for cls, want in (("positive", 744), ("ham", 748)):
    check(S, f"per_generator column sums to {want} for {cls} (so it is not used to plan requests)",
          int(saved_sh.loc[saved_sh["class"] == cls, "per_generator"].sum()), want)
check(S, "added ham scenarios with fewer than three design-fold messages (deviation, see above)",
      sorted(s for s in ADDED["ham"] if Counter(coding.loc[coding.label == 0, "scenario"]).get(s, 0) < 3),
      ["account_security_notice"])
report(S)

## Bước 5: Danh sách yêu cầu sinh tin

Danh sách yêu cầu được commit trước lần gọi API đầu tiên, nên ai cũng kiểm được rằng thứ tự gọi, kịch bản và văn phong của từng tin không bị
chọn lại sau khi thấy kết quả. Luật dựng danh sách (Amendment 1):

- Sáu generator, mỗi generator hai lớp. **Bộ chính**: 750 tin mỗi generator mỗi lớp, chia theo `share_pct` của Bước 4. **Arm B**: 250 tin
  mỗi generator mỗi lớp, chia đều cho các kịch bản trong danh sách ban đầu, không dùng tỷ lệ của design fold. **Hai biến thể prompt**
  (P-min và P-few): mỗi biến thể 250 tin mỗi generator mỗi lớp, cùng tỷ lệ kịch bản với bộ chính. Tổng 18.000 yêu cầu.
- Với P-few, mỗi yêu cầu kèm ba tin thật của design fold làm ví dụ: tối đa ba tin cùng kịch bản, phần còn lại lấy từ các kịch bản khác
  cùng lớp, rút bằng `random.Random(20261101 + 2000000 + request_index)`. Cột `examples` chỉ ghi mã tin (`message_id`), không ghi nội dung.
- Số yêu cầu của mỗi kịch bản là số nguyên, phân bổ bằng **phần dư lớn nhất**: lấy phần nguyên của tổng nhân tỷ lệ, rồi cộng 1 cho các kịch
  bản có phần thập phân lớn nhất cho tới khi đủ tổng. Khi phần thập phân bằng nhau, kịch bản có tên đứng trước theo thứ tự chữ cái được ưu
  tiên.
- Danh sách gốc được sắp theo (generator, bộ, lớp, kịch bản), xáo trộn một lần bằng `random.Random(20261101)`, và `request_index` là vị trí
  sau khi xáo. Nhờ vậy không generator hay kịch bản nào được sinh thành một khối liền.
- Ba trường văn phong (loại người gửi, độ dài, có dấu hay không) được rút đều và độc lập với lớp, bằng `random.Random(20261101 +
  request_index)`, từ danh sách giá trị trong `prompts_v1.md`.

Notebook dựng lại danh sách này và so **từng byte** với `requests.csv` đã commit.

In [ ]:
GENERATORS = ["claude", "gpt", "gemini", "open", "deepseek", "kimi"]   # numbered 1 to 6 in Amendment 1
PER_CELL = {"main": 750, "B": 250, "Pmin": 250, "Pfew": 250}

# Style values: the backticked values on each *_text row of the style table in prompts_v1.md
prompts_md = (EXPECTED / "prompts_v1.md").read_text(encoding="utf-8")
style_part = prompts_md.split("## Style fields", 1)[1]
STYLE = {f: re.findall(r"`([^`]+)`", vals) for f, vals in re.findall(r"^\| (\w+_text) \| (.+?) \|$", style_part, flags=re.M)}
for f, v in STYLE.items():
    print(f, "->", v)

def largest_remainder(weights, total):
    s = sum(weights.values())
    exact = {k: total * w / s for k, w in weights.items()}
    counts = {k: int(v) for k, v in exact.items()}
    for k in sorted(exact, key=lambda k: (-(exact[k] - counts[k]), k))[: total - sum(counts.values())]:
        counts[k] += 1
    return counts

share = {c: dict(zip(g["scenario"], g["share_pct"])) for c, g in saved_sh.groupby("class")}
alloc = {("main", "positive"): largest_remainder(share["positive"], PER_CELL["main"]),
         ("main", "ham"): largest_remainder(share["ham"], PER_CELL["main"]),
         ("B", "positive"): largest_remainder({s: 1 for s in SEED_POS}, PER_CELL["B"]),
         ("B", "ham"): largest_remainder({s: 1 for s in SEED_HAM}, PER_CELL["B"])}
for st in ("Pmin", "Pfew"):                      # prompt variants keep the main set's scenario mix
    for c in ("positive", "ham"):
        alloc[(st, c)] = largest_remainder(share[c], PER_CELL[st])

# P-few examples: design-fold ids by class and scenario, from the committed coding; the two
# positives coded outside the seed list and the mislabelled ham message are left out
POOLS = defaultdict(list)
for mid, lab, sc in zip(coding["message_id"], coding["label"], coding["scenario"]):
    cls = "positive" if lab == 1 else "ham"
    if sc.startswith("other") or sc == "authority_impersonation" or (cls == "ham" and sc == "debt_collection_threat"):
        continue
    POOLS[(cls, sc)].append(mid)
POOLS = {k: sorted(v) for k, v in POOLS.items()}

def draw_examples(cls, scenario, index, n=3):
    # up to three of the same scenario, the rest from the same class's other scenarios
    rng = random.Random(SEED + 2_000_000 + index)
    same = POOLS.get((cls, scenario), [])
    other = sorted(i for (c, sc), ids in POOLS.items() if c == cls and sc != scenario for i in ids)
    k = min(n, len(same))
    return ";".join(rng.sample(same, k) + rng.sample(other, n - k))

reqs = []
for g in GENERATORS:
    for (st, c), counts in alloc.items():
        for s in sorted(counts):
            reqs += [{"generator": g, "set": st, "class": c, "scenario": s} for _ in range(counts[s])]
random.Random(SEED).shuffle(reqs)
for i, r in enumerate(reqs):
    rng = random.Random(SEED + i)
    r.update(request_index=i, replaces="",
             **{f: rng.choice(STYLE[f]) for f in ("sender_text", "length_text", "accent_text")})
    r["examples"] = draw_examples(r["class"], r["scenario"], i) if r["set"] == "Pfew" else ""

FIELDS = ["request_index", "generator", "set", "class", "scenario", "sender_text", "length_text", "accent_text",
          "examples", "replaces"]
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=FIELDS, lineterminator="\n")
w.writeheader()
w.writerows(reqs)
mine_bytes = buf.getvalue().encode("utf-8")
(WORK / "requests.csv").write_bytes(mine_bytes)
saved_bytes = (EXPECTED / "requests.csv").read_bytes()

print("requests:", len(reqs))
print(pd.DataFrame(reqs).groupby(["set", "class"]).size().unstack().to_string())
print("main positive:", alloc[("main", "positive")])

In [ ]:
R = pd.DataFrame(reqs)
S = "5 request list"
check(S, "requests", len(reqs), 18000)
check(S, "every generator x class has 750 main requests",
      set(R[R.set == "main"].groupby(["generator", "class"]).size()), {750})
for st in ("B", "Pmin", "Pfew"):
    check(S, f"every generator x class has 250 {st} requests",
          set(R[R.set == st].groupby(["generator", "class"]).size()), {250})
ex = R.loc[R.set == "Pfew", "examples"].str.split(";")
check(S, "every P-few request lists three distinct design-fold examples",
      bool(ex.map(lambda v: len(set(v)) == 3).all()), True)
fold_ids = set(coding["message_id"])
check(S, "every P-few example is a design-fold message", bool(ex.map(lambda v: set(v) <= fold_ids).all()), True)
check(S, "no request outside P-few carries examples", bool((R.loc[R.set != "Pfew", "examples"] == "").all()), True)
check(S, "style fields are independent of class (same value set in both classes)",
      all(set(R.loc[R["class"] == "positive", f]) == set(R.loc[R["class"] == "ham", f]) for f in STYLE), True)
check(S, "sha256 of the rebuilt list equals the committed requests.csv",
      hashlib.sha256(mine_bytes).hexdigest(), hashlib.sha256(saved_bytes).hexdigest())
report(S)

## Bước 5b: Prompt hoàn chỉnh gửi cho mô hình

Câu chữ của prompt chỉ nằm ở một chỗ, file `prompts_v1.md` đã commit cùng bản đăng ký. Bước này đọc file đó và ghép prompt cho vài yêu cầu
đầu tiên của `requests.csv`, đúng như khi sinh tin. Bước này không gọi mô hình nào.

Mỗi yêu cầu gồm hai phần:
- **System prompt**, giống nhau cho mọi yêu cầu. Nó nói đây là nghiên cứu, yêu cầu viết đúng một tin, và cấm số điện thoại, số tài khoản,
  tên người, đường link hay tên miền có thật. Mọi giá trị cụ thể phải thay bằng thẻ `[NUMBER] [MONEY] [TIME] [DATE] [NAME] [URL]`. Vì vậy
  tin sinh ra không có URL thật nào: chỗ đặt link luôn là `[URL]`.
- **User prompt**, một mẫu năm dòng. Notebook điền vào đó loại tin (theo lớp), mô tả kịch bản (theo bảng kịch bản trong file), và ba trường
  văn phong đã rút ở Bước 5.

Hai biến thể dùng mẫu riêng: P-min chỉ có loại tin và kịch bản, P-few có thêm ba tin thật của design fold đặt trước mẫu chính. Ở đây
ba tin của P-few được in bằng mã tin (`<design-fold message COM_...>`), không in nội dung, để notebook không in tin thật nào.

Mã băm SHA-256 của system prompt được in ra. Khi sinh tin, mỗi dòng log ghi mã băm này, nên ai cũng kiểm được rằng mọi tin được sinh với đúng
system prompt đã đăng ký.

Phần kiểm: mọi kịch bản trong danh sách yêu cầu đều có mô tả trong file, và không prompt nào còn sót chỗ trống `{...}` chưa điền.

In [ ]:
# System prompt and user template: the first two fenced blocks of prompts_v1.md
blocks = re.findall(r"```\n(.*?)\n```", prompts_md, flags=re.S)
SYSTEM_PROMPT = blocks[0]
USER_TEMPLATE = {"main": blocks[1], "B": blocks[1], "Pmin": blocks[2], "Pfew": blocks[3]}

# Class texts: the two backticked values after `class_text`:
CLASS_TEXT = dict(zip(("positive", "ham"),
                      re.findall(r"`class_text`: `([^`]+)`.*?`([^`]+)`", prompts_md)[0]))

# Scenario texts: the rows of the two scenario tables
SCENARIO_TEXT = {}
for cls, head in (("positive", "### Positive"), ("ham", "### Ham")):
    part = prompts_md.split(head, 1)[1].split("###", 1)[0].split("## Style", 1)[0]
    for name, text in re.findall(r"^\| (\w+) \| (.+?) \|$", part, flags=re.M):
        if name != "scenario":
            SCENARIO_TEXT[(cls, name)] = text.strip()

def user_prompt(r):
    # P-few: the examples are shown by message id here, so no corpus text is printed; the
    # generation script puts the messages themselves in their place
    tpl = USER_TEMPLATE[r["set"]]
    if "{examples}" in tpl:
        tpl = tpl.replace("{examples}", "\n".join(f"- <design-fold message {i}>" for i in r["examples"].split(";")))
    return (tpl.replace("{class_text}", CLASS_TEXT[r["class"]])
            .replace("{scenario_text}", SCENARIO_TEXT[(r["class"], r["scenario"])])
            .replace("{sender_text}", r["sender_text"])
            .replace("{length_text}", r["length_text"])
            .replace("{accent_text}", r["accent_text"]))

SYSTEM_SHA256 = hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()
print("system prompt sha256:", SYSTEM_SHA256)
print("\n===== SYSTEM PROMPT (every request) =====")
print(SYSTEM_PROMPT)

# One positive and one ham request from the main set, and one from arm B, in request order
shown = []
for want in (("main", "positive"), ("main", "ham"), ("B", "positive"), ("Pmin", "positive"), ("Pfew", "ham")):
    shown.append(next(r for r in reqs if (r["set"], r["class"]) == want))
for r in shown:
    print(f"\n===== USER PROMPT: request {r['request_index']} ({r['generator']}, set {r['set']}, {r['class']}, "
          f"{r['scenario']}) =====")
    print(user_prompt(r))

system prompt sha256: 3ae3e217dbe69829268f898ef2564a0621b426124df922f870206feadb2ddef6

===== SYSTEM PROMPT (every request) =====
Bạn đang hỗ trợ một nghiên cứu khoa học về phát hiện tin nhắn lừa đảo tiếng Việt. Dữ liệu bạn tạo chỉ dùng để huấn luyện và kiểm tra bộ lọc, không gửi tới ai. Hãy viết đúng MỘT tin nhắn SMS tiếng Việt theo yêu cầu. Quy tắc bắt buộc:
- Không dùng số điện thoại, số tài khoản, tên người, đường link hay tên miền có thật. Thay bằng các thẻ [NUMBER], [MONEY], [TIME], [DATE], [NAME], [URL].
- Viết như một tin nhắn thật mà người Việt nhận được, đúng giọng của người gửi được mô tả.
- Chỉ trả về nội dung tin nhắn, không giải thích, không đặt trong ngoặc kép.

===== USER PROMPT: request 1 (deepseek, set main, positive, gambling_promo) =====
Loại tin: tin nhắn lừa đảo hoặc rác
Kịch bản: quảng cáo nhà cái, cờ bạc, nổ hũ hoặc cá cược, hứa thưởng khi đăng ký hoặc nạp tiền lần đầu
Người gửi: số điện thoại cá nhân
Độ dài: ngắn (dưới 80 ký tự)
Cách viết: tiếng Việt không dấu


In [ ]:
S = "5b prompts"
missing = sorted({(r["class"], r["scenario"]) for r in reqs} - set(SCENARIO_TEXT))
check(S, "every requested scenario has a scenario text", missing, [])
check(S, "scenario texts: 16 positive and 13 ham",
      (sum(c == "positive" for c, _ in SCENARIO_TEXT), sum(c == "ham" for c, _ in SCENARIO_TEXT)), (16, 13))
FIELDS_IN_TEMPLATES = ["{class_text}", "{scenario_text}", "{sender_text}", "{length_text}", "{accent_text}", "{examples}"]
unfilled = sum(any(f in user_prompt(r) for f in FIELDS_IN_TEMPLATES) for r in reqs)
check(S, "user prompts with an unfilled {field}, over all 18,000 requests", unfilled, 0)
check(S, "P-min prompts carry no style field", any("Người gửi" in user_prompt(r) for r in reqs if r["set"] == "Pmin"), False)
check(S, "the system prompt asks for the [URL] placeholder instead of real links", "[URL]" in SYSTEM_PROMPT, True)
report(S)

[5b prompts] 5/5 checks match


## Bước 6 và 7: chưa chạy

Hai bước sau cần dữ liệu sinh, mà nghiên cứu chưa gọi mô hình nào. Chúng sẽ được viết vào notebook khi có dữ liệu:

- **Bước 6, bộ tin sinh**: đọc log sinh tin (mỗi lần gọi một dòng JSON), áp lại bộ lọc theo đúng thứ tự (từ chối, độ dài 20 đến 600 ký tự,
  trùng y hệt, gần trùng), đếm số lần gọi cho mỗi tin được nhận, số tin bị thay thế, chi phí, và cờ ghi nhớ (tin sinh gần trùng một tin thật).
- **Bước 7, phân tích**: chia bốn fold đánh giá theo nhóm, huấn luyện mô hình chính (TF-IDF ký tự 2 đến 5 với hồi quy logistic) cho mọi
  điều kiện, style probe, kiểm ba giả thuyết H1 đến H3 bằng bootstrap theo cụm và hiệu chỉnh Holm.

## Tổng kết

Bảng dưới gom mọi phép kiểm. Cột `match` là `True` khi giá trị notebook tính ra bằng đúng giá trị đã ghi.

In [ ]:
summary = pd.DataFrame(CHECKS)
print(summary.groupby("step", sort=False)["match"].agg(["sum", "count"]).rename(columns={"sum": "match", "count": "checks"}).to_string())
print(f"\nALL: {int(summary.match.sum())}/{len(summary)} checks match")
summary[["step", "check", "match"]]